# 대화 상태 (Conversation State)
모델 상호작용 중 대화 상태를 관리하는 방법을 알아보세요.
OpenAI는 대화 상태를 관리하는 몇 가지 방법을 제공하며, 이는 대화에서 여러 메시지 또는 차례에 걸쳐 정보를 보존하는 데 중요합니다.

각 텍스트 생성 요청은 독립적이고 상태가 지정되지 않지만(Assistants API를 사용하는 경우 제외), 텍스트 생성 요청에 추가 메시지를 매개변수로 제공하여 여러 차례에 걸친 대화를 구현할 수 있습니다. 

### 컨텍스트 창 관리
컨텍스트 창을 이해하면 스레드된 대화를 성공적으로 생성하고 모델 상호작용 전반의 상태를 관리하는 데 도움이 됩니다.   
컨텍스트 창은 단일 요청에서 사용할 수 있는 최대 토큰 수입니다. 이 최대 토큰 수에는 입력, 출력 및 추론 토큰이 포함됩니다. 모델의 컨텍스트 창에 대한 자세한 내용은 모델 세부 정보를 참조하세요. 

다음 토큰 수가 컨텍스트 창 총계에 적용됩니다.
- 입력 토큰( Responses APIinput 에 대한 배열 에 포함하는 입력)  
- 출력 토큰(프롬프트에 대한 응답으로 생성된 토큰)  
- 추론 토큰(모델이 응답을 계획하는 데 사용됨)
  
컨텍스트 창 제한을 초과하여 생성된 토큰은 API 응답에서 잘릴 수 있습니다.

<img src="https://i.imgur.com/WcWPlmZ.png" width=500 />

In [7]:
from dotenv import load_dotenv
load_dotenv() # read local .env file

True

In [8]:
from openai import OpenAI
import json

client = OpenAI()

Model = "gpt-5-nano"

### 대화 상태 관리
response API를 사용하면 대화 상태를 자동으로 더 쉽게 관리할 수 있으므로 대화가 진행될 때마다 수동으로 입력을 전달할 필요가 없습니다.  
Responses API는 상태를 저장하므로 대화 전반의 컨텍스트 관리가 간단한 매개변수로 가능합니다.

In [3]:
response = client.responses.create(
    model=Model,
    input="재미있는 농담을 해줘.",
)
print(response.output_text)

second_response = client.responses.create(
    model=Model,
    previous_response_id=response.id,
    input=[{"role": "user", "content": "그게 왜 재미 있는지 설명해줘"}],
)
print(second_response.output_text)

좋아요! 어떤 분위기의 농담을 원하시나요? 선택해주시면 그 스타일로 3~5개 바로 드릴게요.
- 아재개그
- 말장난(언어유희)
- 짧은 상황극
- 동물농담
- 기타(원하는 스타일 적어주셔도 됨)

원하시면 지금 바로 하나의 짧은 농담도 바로 드릴게요. 어떤 스타일이 좋을지 알려주시겠어요?
좋아요. 구체적으로 어떤 농담을 예로 들었는지 모르겠지만, 일반적으로 왜 농담이 재미있는지 핵심 포인트를 정리해볼게요.

주요 재미 요인
- 기대와 현실의 차이(이상과 현실의 불일치): 듣는 이가 예상한 방향과 다른 결말이 나올 때 생기는 놀라움이 웃음을 만듭니다.
- 언어유희/말장난: 단어나 소리의 닮음, 다의어를 이용한 의외의 연결이 웃음을 만들어냅니다.
- 의도된 오해와 재해석: 말의 표면적 의미를 벗어나 숨은 의도나 상황을 확인했을 때 재해석으로 웃김이 생깁니다.
- 과장(하이퍼볼)과 극단화: 현실에서 불가능한 상황을 지나치게 크게 그려 웃음을 유발합니다.
- 상황 맥락과 공감: 청자와의 공유된 규범이나 일상적 경험을 바탕으로 할 때 더 쉽게 웃음이 납니다.
- 간결함과 타이밍: 짧고 간결한 한 문장, 혹은 한 번의 탁 트인 순간에 웃음을 주는 타이밍이 중요합니다.
- 자기비하나 타인에 대한 해학적 관용: 스스로를 가볍게 비하하거나 상대를 배려하는 선에서의 웃음은 친근감을 줍니다.

원하시면 지금 바로 하나의 예시를 만들어 드리고, 그에 대한 재미 포인트도 함께 설명해 드릴게요. 어떤 스타일로 원하시나요?
- 아재개그
- 말장난(언어유희)
- 짧은 상황극
- 동물농담
- 기타(원하시는 분위기 적어주시면 맞춰드림)

또는 특정 농담이 있다면 보내주시면 그 농담의 재미 포인트를 분석해 드리겠습니다.


### 문서/텍스트를 기반으로 챗봇이 답변하도록 하기

짧은 안내 문서(FAQ)를 챗봇에게 넘겨주고, `previous_response_id` 로 대화 맥락을 이어가는 **간단한 예제**입니다.

핵심은 두 가지입니다.
1. `instructions` 로 챗봇의 역할과 규칙을 정한다.
2. 참고할 문서를 입력으로 넘겨주면, 챗봇이 그 문서 범위 안에서만 답한다.

In [4]:
# 챗봇이 참고할 짧은 안내 문서 (FAQ)
dataset = """[카페 무브(Cafe Move) 안내]
- 영업시간: 매일 오전 9시 ~ 오후 10시 (연중무휴)
- 위치: 서울시 강남구 테헤란로 123, 1층
- 대표 메뉴: 아메리카노 4,500원 / 카페라떼 5,000원 / 바닐라라떼 5,500원
- 와이파이: 무료 제공 (비밀번호는 영수증 하단에 표시)
- 주차: 1시간 무료, 이후 10분당 1,000원
- 반려동물: 소형견에 한해 동반 가능
"""

In [6]:
# instructions 로 역할/규칙을 정하고, 첫 입력으로 안내 문서를 넘겨줍니다.
instructions = """
당신은 '카페 무브'의 친절한 안내 도우미입니다.
사용자가 제공한 안내 문서의 내용만 근거로 답변하세요.
문서에 없는 내용은 "죄송합니다. 해당 정보는 없습니다."라고만 답하세요.
첫 메시지에서는 문서 내용을 나열하지 말고 "안녕하세요! 카페 무브입니다. 무엇을 도와드릴까요?"라고만 인사하세요.
"""

response = client.responses.create(
    model="gpt-5-mini",
    instructions=instructions,
    input=[{"role": "user", "content": dataset}],
)
print("assistant:", response.output_text)

# previous_response_id 로 이전 대화 맥락을 이어갑니다.
while True:
    user_input = input("사용자 입력 (종료: quit): ")
    if user_input.lower() == "quit":
        break

    response = client.responses.create(
        model=Model,
        previous_response_id=response.id,
        input=[{"role": "user", "content": user_input}],
    )
    print("assistant:", response.output_text)

assistant: 안녕하세요! 카페 무브입니다. 무엇을 도와드릴까요?


사용자 입력 (종료: quit):  카페 영업시간이 언제부터 언제까지야?


assistant: 영업시간은 매일 오전 9시부터 오후 10시까지이며, 연중무휴로 운영됩니다.


사용자 입력 (종료: quit):  quit


------------------
(대화 예시)
```
assistant: 안녕하세요! 카페 무브입니다. 무엇을 도와드릴까요?
사용자 입력 (종료: quit):  영업시간이 어떻게 되나요?
assistant: 카페 무브는 매일 오전 9시부터 오후 10시까지 연중무휴로 운영합니다.
사용자 입력 (종료: quit):  주차도 되나요?
assistant: 네, 1시간 무료 주차가 가능하며 이후에는 10분당 1,000원입니다.
사용자 입력 (종료: quit):  강아지 데려가도 되나요?
assistant: 소형견에 한해 동반 입장이 가능합니다.
사용자 입력 (종료: quit):  배달도 되나요?
assistant: 죄송합니다. 해당 정보는 없습니다.
사용자 입력 (종료: quit):  quit
```

> `previous_response_id` 덕분에 "주차도 되나요?", "강아지 데려가도 되나요?" 처럼 짧게 물어도 이전 맥락(카페 무브)을 기억하고 답합니다.

## 스트리밍으로 응답 받기 (Streaming)

기본적으로 OpenAI API에 요청을 보내면 모델이 **전체 출력을 다 생성한 후** 단일 HTTP 응답으로 한 번에 돌려줍니다. 긴 답변을 생성할 경우 사용자는 응답을 기다리는 동안 빈 화면을 보게 됩니다.

스트리밍 응답을 사용하면 모델이 출력을 생성하는 도중에 **시작 부분부터 조각(chunk) 단위로** 받아서 바로 처리하거나 화면에 출력할 수 있습니다.

응답 스트리밍을 시작하려면 `client.responses.create(...)` 요청에 `stream=True` 를 설정하면 됩니다.

> 스트리밍은 특히 **대화가 오가는 챗봇** 맥락에서 그 효과가 가장 잘 드러납니다. 토큰이 실시간으로 찍히는 경험이 응답 대기 체감 시간을 크게 줄여 주기 때문입니다. 아래에서 앞서 다룬 대화 맥락 관리와 스트리밍을 함께 적용해 봅니다.

In [9]:
stream = client.responses.create(
    model=Model,
    input=[
        {
            "role": "user",
            "content": "성경의 창세기 시작 부분을 1000 글자내에서 알려줘.",  # 사용자 입력 프롬프트
        },
    ],
    stream=True,     # 스트리밍 응답 활성화 (조각 단위로 응답 받기)
)

# 응답이 조각(chunk) 단위로 도착할 때마다 출력
for chunk in stream:
    if chunk.type == 'response.output_text.delta':  # 응답 조각이 텍스트 델타(추가분)인 경우만 처리
        print(chunk.delta, end="")       # 출력된 텍스트 델타를 이어서 출력 (줄바꿈 없이)

다음은 창세기 시작 부분을 1000자 이내로 이해하기 쉽게 정리한 요약입니다.

기 1장: 태초에 하나님이 천지와 모든 것을 창조하시니, 땅은 아직 형체 없고 공허하며 어둠만 깊음 위에 있었다. 하나님의 신이 수면 위를 운행하심. 하나님이 말씀하시되 빛이 있으라 하시자 빛이 생겨나고 빛과 어둠을 나누어 낮과 밤이라 부름. 이틀에 걸쳐 하늘(궁창), 물과 땅의 분리 등을 창조. 셋째 날에는 땅에서 땅의 식물과 나무가 자라나게 하심. 넷째 날에는 해와 달과 별을 만들어 낮과 밤, 계절을 다스리게 하심. 다섯째 날에는 물고기와 새를, 여섯째 날에는 들의 짐승과 특히 사람을 하나님의 형상대로 창조하셨다. 남자와 여자에 대해 땅을 다스리고, 하나님은 보시기에 좋았다고 하셨다. 여섯 일을 마치고 하나님은 안식하셨고, 이것이 창조의 완성 및 안식의 시작이다.

필요하신 형식 선택
메인 버전인 KJV의 시작 부분을 그대로 제공해 드릴 수 있습니다.
. 어떤 방식으로 드릴까요? 특정 구절의 해설도 가능합니다

### 스트리밍 + 대화 맥락을 함께 적용한 챗봇

이제 앞에서 배운 **대화 맥락 유지**와 방금 본 **스트리밍**을 하나로 합쳐 봅니다.

- 대화 히스토리를 `list` 로 직접 관리하면서 (`conversation_history`)
- 매 응답을 `stream=True` 로 받아 **토큰이 생성되는 즉시 화면에 출력**합니다.

이렇게 하면 사용자는 응답이 완성될 때까지 기다리지 않고, 글자가 실시간으로 찍히는 자연스러운 챗봇 경험을 하게 됩니다.

In [10]:
# 대화 히스토리를 list로 관리하면서, 응답을 스트리밍으로 출력하는 챗봇
# - input 에 conversation_history(list) 를 통째로 넘겨 맥락을 유지합니다.
# - stream=True 로 토큰이 생성되는 즉시 화면에 출력합니다.

conversation_history = []

print("=== 스트리밍 챗봇 (대화 맥락 유지) ===")
print("종료하려면 'quit' 또는 'exit'를 입력하세요.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["quit", "exit"]:
        print("챗봇을 종료합니다.")
        break

    # 사용자 입력을 히스토리에 추가
    conversation_history.append({"role": "user", "content": user_input})

    # 지금까지의 대화 전체를 input 으로 넘기고, 스트리밍으로 응답 받기
    stream = client.responses.create(
        model=Model,
        input=conversation_history,
        stream=True,
    )

    print("Assistant: ", end="", flush=True)
    assistant_message = ""
    for chunk in stream:
        if chunk.type == "response.output_text.delta":
            print(chunk.delta, end="", flush=True)  # 토큰이 도착하는 즉시 출력
            assistant_message += chunk.delta
    print("\n")

    # 완성된 응답을 히스토리에 추가하여 다음 차례의 맥락으로 사용
    conversation_history.append({"role": "assistant", "content": assistant_message})

=== 스트리밍 챗봇 (대화 맥락 유지) ===
종료하려면 'quit' 또는 'exit'를 입력하세요.



You:  안녕, 난 길동이야


Assistant: 안녕 길동! 반가워요. 오늘은 무엇을 도와줄까요? 

- 대화 연습이나 한국어 공부
- 번역이나 문법 설명
- 정보 찾기나 간단한 문제 해결
- 취미나 관심사에 맞춘 주제 대화

원한다면 간단한 자기소개를 같이 만들어 보거나 오늘의 표현부터 시작해볼 수도 있어요. 예를 들어 이렇게 자연스럽게 다듬어볼 수 있어요:
“안녕하세요. 저는 길동이라고 합니다. 한국어를 배우고 있습니다. 잘 부탁드립니다.”

어떤 걸 먼저 해볼까요?



You:  창세기 시작 부분을 1000자 이내로 읊어줘


Assistant: 요청하신 부분의 원문을 그대로 길게 옮겨 드리지는 못하지만, 시작 부분의 내용을 1000자 이내로 간단히 요약해 드릴게요.

창세기 시작은 “태초에 하나님이 천지를 창조하시니라”로 시작합니다. 땅은 한때 형태가 없고 공허하며 어둠이 깊은 바다 위에 덮여 있었습니다. 하나님의 신은 물 위를 흘러다녔고, 하나님은 빛을 창조하시며 “빛이 있으라”고 말씀하셨습니다. 빛은 어둠과 나뉘어 낮과 밤이 되었고, 이것이 첫째 날의 시작이었습니다. 둘째 날에는 하늘을 만들어 물을 위와 아래로 나누셨습니다. 셋째 날에는 땅이 드러나고 바다와 육지가 나뉘며 땅이 식물과 나무를 내었습니다. 넷째 날에는 해와 달과 별을 만드시고 해와 달이 낮과 밤을 구분하고 계절을 이루게 하셨습니다. 다섯째 날에는 바다의 생물과 하늘의 새를 만드셨고, 여섯째 날에는 땅의 동물들을 창조하시고, 하나님의 형상대로 사람을 창조하셨습니다. 사람들에게 땅을 다스리고 번성하라 축복하시며 먹을 것을 주셨고, 일곱째 날에는 모든 일을 마치고 쉬셨고 거룩하게 되셨습니다.

원하시면 특정 버전의 짧은 구절을 인용해 드리거나, 더 구체적으로 바꿔 요약해 드릴게요. 어떤 버전으로 원하시나요?



You:  quit


챗봇을 종료합니다.


### 실습 문제: 대화 흐름과 상태 유지

`OpenAI Responses API`의 `previous_response_id` 를 사용하여, **이전 응답을 기억하고 연관성 있게 대화를 이어가는** 디지털 헬스 상담 챗봇을 만들어 보세요.

### 주어진 초기 설정

- 사용자 첫 질문: `"요즘 소화가 잘 안돼요. 어떻게 해야 할까요?"`
- 이어지는 질문: `"그럼 매운 음식은 피해야 하나요?"`

### 구현 조건

1. Responses API를 사용하여 첫 번째 질문에 답변을 생성하세요.
2. `previous_response_id` 를 활용하여 두 번째 질문을 **동일한 컨텍스트 흐름**으로 이어서 응답하도록 하세요.
3. 두 번째 응답은 **첫 번째 대답을 기억한 상태**에서 이어지는 조언처럼 들려야 합니다.
4. 응답 마지막에 `"다른 증상은 없으신가요?"` 와 같이 자연스럽게 다음 질문을 유도하세요.

### 테스트 입력 예시

- 첫 질문: `"요즘 소화가 잘 안돼요. 어떻게 해야 할까요?"`
- 후속: `"그럼 매운 음식은 피해야 하나요?"` → 앞의 소화 맥락을 이어 매운 음식 조언